In [19]:
import os
import json
import pickle
import datetime

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report

from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications import MobileNetV2, EfficientNetB0
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess

In [20]:
SEEDS = [42, 123, 2024, 3407, 777]

BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 5
LEARNING_RATE = 1e-5
FINE_TUNE_LAYERS = 20
AUTOTUNE = tf.data.AUTOTUNE

CANONICAL_CLASSES = ["Early Blight", "Late Blight", "Healthy"]
NUM_CLASSES = len(CANONICAL_CLASSES)

SOURCE_DATASET_DIR = "/kaggle/input/datasets/faysalmiah1721758/potato-dataset"
TARGET_DATASET_DIR = "/kaggle/input/datasets/shahadhossin567r7455/potato-leaf-disease-dataset/Potato Leaf DIsease"

RUN_TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
OUTPUT_DIR = f"cross_dataset_transfer_{RUN_TIMESTAMP}"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "histories"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "reports"), exist_ok=True)

print("Output directory:", OUTPUT_DIR)

Output directory: cross_dataset_transfer_20260530-210441


In [21]:
CLASS_FOLDER_CANDIDATES = {
    "Early Blight": [
        "Early Blight",
        "Early_Blight",
        "Potato___Early_blight",
        "Potato___Early_Blight"
    ],
    "Late Blight": [
        "Late Blight",
        "Late_Blight",
        "Potato___Late_blight",
        "Potato___Late_Blight"
    ],
    "Healthy": [
        "Healthy",
        "healthy",
        "Potato___healthy",
        "Potato___Healthy"
    ]
}


def find_class_folder(dataset_dir, canonical_class):
    available_folders = [
        folder for folder in os.listdir(dataset_dir)
        if os.path.isdir(os.path.join(dataset_dir, folder))
    ]

    for candidate in CLASS_FOLDER_CANDIDATES[canonical_class]:
        if candidate in available_folders:
            return os.path.join(dataset_dir, candidate)

    raise ValueError(
        f"Could not find folder for class '{canonical_class}' in {dataset_dir}. "
        f"Available folders: {available_folders}"
    )


def collect_paths_and_labels(dataset_dir):
    valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

    file_paths = []
    labels = []

    for label_index, class_name in enumerate(CANONICAL_CLASSES):
        class_dir = find_class_folder(dataset_dir, class_name)

        for fname in os.listdir(class_dir):
            if fname.lower().endswith(valid_extensions):
                file_paths.append(os.path.join(class_dir, fname))
                labels.append(label_index)

    file_paths = np.array(file_paths)
    labels = np.array(labels)

    print("\nDataset:", dataset_dir)
    print("Total images:", len(labels))

    unique, counts = np.unique(labels, return_counts=True)
    for idx, count in zip(unique, counts):
        print(f"{CANONICAL_CLASSES[idx]}: {count}")

    return file_paths, labels

In [22]:
source_paths, source_labels = collect_paths_and_labels(SOURCE_DATASET_DIR)
target_paths, target_labels = collect_paths_and_labels(TARGET_DATASET_DIR)


Dataset: /kaggle/input/datasets/faysalmiah1721758/potato-dataset
Total images: 2152
Early Blight: 1000
Late Blight: 1000
Healthy: 152

Dataset: /kaggle/input/datasets/shahadhossin567r7455/potato-leaf-disease-dataset/Potato Leaf DIsease
Total images: 1500
Early Blight: 500
Late Blight: 500
Healthy: 500


In [23]:
def make_splits(file_paths, labels, seed):
    X_train, X_temp, y_train, y_temp = train_test_split(
        file_paths,
        labels,
        test_size=0.20,
        stratify=labels,
        random_state=seed
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=seed
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

In [24]:
def load_tf_dataset(file_paths, labels, image_size, preprocess_fn, augment=False, shuffle=True, seed=None):
    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))

    def process(path, label):
        image = tf.io.read_file(path)
        image = tf.image.decode_image(image, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])

        image = tf.image.resize(image, image_size)
        image = tf.cast(image, tf.float32)

        if augment:
            image = tf.image.random_flip_left_right(image)
            image = tf.image.random_brightness(image, max_delta=25.5)
            image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
            image = tf.clip_by_value(image, 0.0, 255.0)

        image = preprocess_fn(image)
        label = tf.one_hot(label, depth=NUM_CLASSES)

        return image, label

    ds = ds.map(process, num_parallel_calls=AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(file_paths),
            seed=seed,
            reshuffle_each_iteration=True
        )

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

    return ds

In [25]:
def cnn_preprocess(image):
    return image / 255.0


MODEL_CONFIGS = {
    "Lightweight CNN": {
        "image_size": (256, 256),
        "preprocess_fn": cnn_preprocess
    },
    "MobileNetV2": {
        "image_size": (224, 224),
        "preprocess_fn": mobilenet_preprocess
    },
    "EfficientNetB0": {
        "image_size": (224, 224),
        "preprocess_fn": efficientnet_preprocess
    }
}

In [26]:
def build_improved_cnn(input_shape=(256, 256, 3), num_classes=3):
    inputs = tf.keras.Input(shape=input_shape)

    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), activation='relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name="LightweightCNN")
    return model


def build_mobilenet_v2(input_shape=(224, 224, 3), num_classes=3):
    inputs = tf.keras.Input(shape=input_shape)

    backbone = MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs
    )

    backbone.trainable = True

    for layer in backbone.layers[:-FINE_TUNE_LAYERS]:
        layer.trainable = False

    x = backbone.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs, name="MobileNetV2_transfer")


def build_efficientnet_b0(input_shape=(224, 224, 3), num_classes=3):
    inputs = tf.keras.Input(shape=input_shape)

    backbone = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs
    )

    backbone.trainable = True

    for layer in backbone.layers[:-FINE_TUNE_LAYERS]:
        layer.trainable = False

    x = backbone.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs, name="EfficientNetB0_transfer")


MODEL_BUILDERS = {
    "Lightweight CNN": build_improved_cnn,
    "MobileNetV2": build_mobilenet_v2,
    "EfficientNetB0": build_efficientnet_b0
}

In [27]:
def get_class_weights(y_train):
    class_weight_vals = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(NUM_CLASSES),
        y=y_train
    )

    return dict(enumerate(class_weight_vals))

In [28]:
def run_cross_dataset_experiment(model_name, seed):
    print("\n" + "=" * 80)
    print(f"Model: {model_name} | Seed: {seed}")
    print("=" * 80)

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)

    image_size = MODEL_CONFIGS[model_name]["image_size"]
    preprocess_fn = MODEL_CONFIGS[model_name]["preprocess_fn"]

    source_train, source_val, source_test, y_source_train, y_source_val, y_source_test = make_splits(
        source_paths,
        source_labels,
        seed
    )

    target_train, target_val, target_test, y_target_train, y_target_val, y_target_test = make_splits(
        target_paths,
        target_labels,
        seed
    )

    source_train_ds = load_tf_dataset(source_train, y_source_train, image_size, preprocess_fn, augment=True, shuffle=True, seed=seed)
    source_val_ds = load_tf_dataset(source_val, y_source_val, image_size, preprocess_fn, augment=False, shuffle=False)
    source_test_ds = load_tf_dataset(source_test, y_source_test, image_size, preprocess_fn, augment=False, shuffle=False)

    target_train_ds = load_tf_dataset(target_train, y_target_train, image_size, preprocess_fn, augment=True, shuffle=True, seed=seed)
    target_val_ds = load_tf_dataset(target_val, y_target_val, image_size, preprocess_fn, augment=False, shuffle=False)
    target_test_ds = load_tf_dataset(target_test, y_target_test, image_size, preprocess_fn, augment=False, shuffle=False)

    model = MODEL_BUILDERS[model_name](
        input_shape=(image_size[0], image_size[1], 3),
        num_classes=NUM_CLASSES
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    safe_model_name = model_name.replace(" ", "_")
    run_name = f"{safe_model_name}_seed_{seed}"

    source_model_path = os.path.join(OUTPUT_DIR, "models", f"{run_name}_source.keras")
    finetuned_model_path = os.path.join(OUTPUT_DIR, "models", f"{run_name}_target_finetuned.keras")

    source_history_path = os.path.join(OUTPUT_DIR, "histories", f"{run_name}_source_history.pkl")
    target_history_path = os.path.join(OUTPUT_DIR, "histories", f"{run_name}_target_finetune_history.pkl")

    source_callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True
        ),
        ModelCheckpoint(
            source_model_path,
            monitor="val_loss",
            save_best_only=True
        )
    ]

    source_history = model.fit(
        source_train_ds,
        validation_data=source_val_ds,
        epochs=EPOCHS,
        callbacks=source_callbacks,
        class_weight=get_class_weights(y_source_train),
        verbose=1
    )

    with open(source_history_path, "wb") as f:
        pickle.dump(source_history.history, f)

    source_eval = evaluate_model(model, source_test_ds)
    zero_shot_eval = evaluate_model(model, target_test_ds)

    target_callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True
        ),
        ModelCheckpoint(
            finetuned_model_path,
            monitor="val_loss",
            save_best_only=True
        )
    ]

    target_history = model.fit(
        target_train_ds,
        validation_data=target_val_ds,
        epochs=EPOCHS,
        callbacks=target_callbacks,
        class_weight=get_class_weights(y_target_train),
        verbose=1
    )

    with open(target_history_path, "wb") as f:
        pickle.dump(target_history.history, f)

    finetuned_eval = evaluate_model(model, target_test_ds)

    source_acc = source_eval["accuracy"]
    zero_acc = zero_shot_eval["accuracy"]
    fine_acc = finetuned_eval["accuracy"]

    result = {
        "model": model_name,
        "seed": seed,

        "source_loss": source_eval["loss"],
        "source_accuracy": source_acc,
        "source_macro_f1": source_eval["macro_f1"],
        "source_weighted_f1": source_eval["weighted_f1"],

        "zero_shot_loss": zero_shot_eval["loss"],
        "zero_shot_accuracy": zero_acc,
        "zero_shot_macro_f1": zero_shot_eval["macro_f1"],
        "zero_shot_weighted_f1": zero_shot_eval["weighted_f1"],

        "finetuned_loss": finetuned_eval["loss"],
        "finetuned_accuracy": fine_acc,
        "finetuned_macro_f1": finetuned_eval["macro_f1"],
        "finetuned_weighted_f1": finetuned_eval["weighted_f1"],

        "zero_shot_drop": source_acc - zero_acc,
        "finetuned_drop": source_acc - fine_acc,
        "zero_shot_ratio": zero_acc / source_acc if source_acc > 0 else np.nan,
        "finetuned_ratio": fine_acc / source_acc if source_acc > 0 else np.nan,

        "source_model_path": source_model_path,
        "finetuned_model_path": finetuned_model_path,
        "source_history_path": source_history_path,
        "target_history_path": target_history_path
    }

    print("\nResult:")
    print(result)

    return result

In [30]:
from sklearn.metrics import f1_score

def evaluate_model(model, dataset):
    y_true = []
    y_pred = []

    loss, acc = model.evaluate(dataset, verbose=0)

    for images, labels in dataset:
        probs = model.predict(images, verbose=0)
        y_true.extend(np.argmax(labels.numpy(), axis=1))
        y_pred.extend(np.argmax(probs, axis=1))

    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")

    return {
        "loss": loss,
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "y_true": y_true,
        "y_pred": y_pred
    }

In [31]:
all_transfer_results = []

for model_name in ["Lightweight CNN", "MobileNetV2", "EfficientNetB0"]:
    for seed in SEEDS:
        result = run_cross_dataset_experiment(model_name, seed)
        all_transfer_results.append(result)

per_seed_df = pd.DataFrame(all_transfer_results)

per_seed_csv = os.path.join(OUTPUT_DIR, "cross_dataset_transfer_per_seed_results.csv")
per_seed_df.to_csv(per_seed_csv, index=False)

print("Saved per-seed results to:", per_seed_csv)
per_seed_df


Model: Lightweight CNN | Seed: 42
Epoch 1/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 12s 101ms/step - accuracy: 0.4648 - loss: 1.0968 - val_accuracy: 0.4651 - val_loss: 1.0930
Epoch 2/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 43ms/step - accuracy: 0.4648 - loss: 1.0929 - val_accuracy: 0.4651 - val_loss: 1.0891
Epoch 3/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 44ms/step - accuracy: 0.4724 - loss: 1.0871 - val_accuracy: 0.4651 - val_loss: 1.0788
Epoch 4/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 44ms/step - accuracy: 0.4863 - loss: 1.0768 - val_accuracy: 0.5256 - val_loss: 1.0708
Epoch 5/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - accuracy: 0.5107 - loss: 1.0577 - val_accuracy: 0.5302 - val_loss: 1.0389
Epoch 6/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - accuracy: 0.5148 - loss: 1.0146 - val_accuracy: 0.5209 - val_loss: 1.0029
Epoch 7/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 44ms/step - accuracy: 0.5026 - loss: 0.9407 - val_accuracy: 0.5302 - val_loss: 0.9179
Epoch 8/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 43ms/step - accuracy: 0.5334 - lo

/tmp/ipykernel_58/4031514531.py:33: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  backbone = MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/50


2026-05-30 22:01:46.815188: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-30 22:01:47.012639: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


53/54 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.4108 - loss: 1.0937

...

2026-05-30 22:26:14.022562: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-30 22:26:14.232247: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


54/54 ━━━━━━━━━━━━━━━━━━━━ 56s 535ms/step - accuracy: 0.4956 - loss: 1.0269 - val_accuracy: 0.5116 - val_loss: 0.9654
Epoch 2/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - accuracy: 0.6880 - loss: 0.8366 - val_accuracy: 0.7674 - val_loss: 0.7647
Epoch 3/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 5s 54ms/step - accuracy: 0.7978 - loss: 0.6853 - val_accuracy: 0.8372 - val_loss: 0.6204
Epoch 4/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 5s 53ms/step - accuracy: 0.8565 - loss: 0.5762 - val_accuracy: 0.8930 - val_loss: 0.5121
Epoch 5/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 53ms/step - accuracy: 0.8815 - loss: 0.4838 - val_accuracy: 0.9302 - val_loss: 0.4313
Epoch 6/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 53ms/step - accuracy: 0.9047 - loss: 0.4218 - val_accuracy: 0.9349 - val_loss: 0.3687
Epoch 7/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 53ms/step - accuracy: 0.9152 - loss: 0.3693 - val_accuracy: 0.9535 - val_loss: 0.3176
Epoch 8/50
54/54 ━━━━━━━━━━━━━━━━━━━━ 4s 52ms/step - accuracy: 0.9250 - loss: 0.3158 - val_accuracy: 0.9581 - val_loss: 

2026-05-30 22:30:41.653805: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-30 22:30:41.859494: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


38/38 ━━━━━━━━━━━━━━━━━━━━ 17s 425ms/step - accuracy: 0.8817 - loss: 0.2799 - val_accuracy: 0.9667 - val_loss: 0.1372
Epoch 2/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 57ms/step - accuracy: 0.9425 - loss: 0.1648 - val_accuracy: 0.9733 - val_loss: 0.1047
Epoch 3/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - accuracy: 0.9675 - loss: 0.1189 - val_accuracy: 0.9667 - val_loss: 0.0915
Epoch 4/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 59ms/step - accuracy: 0.9750 - loss: 0.0921 - val_accuracy: 0.9667 - val_loss: 0.0831
Epoch 5/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 59ms/step - accuracy: 0.9833 - loss: 0.0850 - val_accuracy: 0.9667 - val_loss: 0.0772
Epoch 6/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - accuracy: 0.9733 - loss: 0.0859 - val_accuracy: 0.9667 - val_loss: 0.0741
Epoch 7/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 60ms/step - accuracy: 0.9908 - loss: 0.0640 - val_accuracy: 0.9667 - val_loss: 0.0710
Epoch 8/50
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - accuracy: 0.9867 - loss: 0.0678 - val_accuracy: 0.9667 - val_loss: 

,model,seed,source_loss,source_accuracy,source_macro_f1,source_weighted_f1,zero_shot_loss,zero_shot_accuracy,zero_shot_macro_f1,zero_shot_weighted_f1,...,finetuned_macro_f1,finetuned_weighted_f1,zero_shot_drop,finetuned_drop,zero_shot_ratio,finetuned_ratio,source_model_path,finetuned_model_path,source_history_path,target_history_path
0,Lightweight CNN,42,0.318515,0.865741,0.847770,0.866348,0.833896,0.706667,0.711396,0.711396,...,0.865391,0.865391,0.159074,-0.000926,0.816257,1.001070,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/histori...,cross_dataset_transfer_20260530-210441/histori...
1,Lightweight CNN,123,0.205209,0.912037,0.874069,0.914839,1.200278,0.653333,0.650815,0.650815,...,0.906754,0.906754,0.258704,0.005370,0.716345,0.994112,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/histori...,cross_dataset_transfer_20260530-210441/histori...
2,Lightweight CNN,2024,0.266094,0.935185,0.909735,0.936036,0.979063,0.680000,0.677169,0.677169,...,0.839691,0.839691,0.255185,0.095185,0.727129,0.898218,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/histori...,cross_dataset_transfer_20260530-210441/histori...
3,Lightweight CNN,3407,0.306253,0.884259,0.826766,0.889026,0.968668,0.720000,0.715678,0.715678,...,0.873644,0.873644,0.164259,0.010926,0.814241,0.987644,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/histori...,cross_dataset_transfer_20260530-210441/histori...
4,Lightweight CNN,777,0.326228,0.893519,0.874404,0.895361,1.188257,0.533333,0.503200,0.503200,...,0.906242,0.906242,0.360185,-0.013148,0.596891,1.014715,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/histori...,cross_dataset_transfer_20260530-210441/histori...
5,MobileNetV2,42,0.049798,0.976852,0.952828,0.977693,0.455985,0.826667,0.822802,0.822802,...,0.911107,0.911107,0.150185,0.063519,0.846256,0.934976,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/histori...,cross_dataset_transfer_20260530-210441/histori...
6,MobileNetV2,123,0.038023,0.986111,0.966352,0.986601,0.314294,0.853333,0.847568,0.847568,...,0.979998,0.979998,0.132778,0.006111,0.865352,0.993803,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/histori...,cross_dataset_transfer_20260530-210441/histori...
7,MobileNetV2,2024,0.017994,0.990741,0.975833,0.990741,0.283290,0.900000,0.898649,0.898649,...,0.925768,0.925768,0.090741,0.064074,0.908411,0.935327,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/histori...,cross_dataset_transfer_20260530-210441/histori...
8,MobileNetV2,3407,0.063688,0.976852,0.937945,0.976536,0.272531,0.893333,0.890580,0.890580,...,0.993333,0.993333,0.083519,-0.016481,0.914502,1.016872,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/histori...,cross_dataset_transfer_20260530-210441/histori...
9,MobileNetV2,777,0.028468,0.995370,0.988224,0.995429,0.480506,0.813333,0.810187,0.810187,...,0.939284,0.939284,0.182037,0.055370,0.817116,0.944372,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/models/...,cross_dataset_transfer_20260530-210441/histori...,cross_dataset_transfer_20260530-210441/histori...


In [34]:
def mean_std_percent(series):
    mean = series.mean() * 100
    std = series.std(ddof=0) * 100
    return f"{mean:.2f} ± {std:.2f}"


def mean_std_ratio(series):
    mean = series.mean()
    std = series.std(ddof=0)
    return f"{mean:.4f} ± {std:.4f}"


summary_rows = []

for model_name, group in per_seed_df.groupby("model"):
    summary_rows.append({
        "Model": model_name,
        "Source Acc (%)": mean_std_percent(group["source_accuracy"]),
        "Zero-shot Acc (%)": mean_std_percent(group["zero_shot_accuracy"]),
        "Fine-tuned Acc (%)": mean_std_percent(group["finetuned_accuracy"]),
        "Zero-shot Drop (%)": mean_std_percent(group["zero_shot_drop"]),
        "Fine-tuned Drop (%)": mean_std_percent(group["finetuned_drop"]),
        "Zero-shot Ratio": mean_std_ratio(group["zero_shot_ratio"]),
        "Fine-tuned Ratio": mean_std_ratio(group["finetuned_ratio"]),
        "Source Macro F1": mean_std_ratio(group["source_macro_f1"]),
        "Zero-shot Macro F1": mean_std_ratio(group["zero_shot_macro_f1"]),
        "Fine-tuned Macro F1": mean_std_ratio(group["finetuned_macro_f1"]),
        "Source Weighted F1": mean_std_ratio(group["source_weighted_f1"]),
        "Zero-shot Weighted F1": mean_std_ratio(group["zero_shot_weighted_f1"]),
        "Fine-tuned Weighted F1": mean_std_ratio(group["finetuned_weighted_f1"])
    })

summary_df = pd.DataFrame(summary_rows)

summary_path = os.path.join(OUTPUT_DIR, "cross_dataset_transfer_results.csv")
summary_df.to_csv(summary_path, index=False)

print("Saved summary results to:", summary_path)
summary_df

Saved summary results to: cross_dataset_transfer_20260530-210441/cross_dataset_transfer_results.csv


,Model,Source Acc (%),Zero-shot Acc (%),Fine-tuned Acc (%),Zero-shot Drop (%),Fine-tuned Drop (%),Zero-shot Ratio,Fine-tuned Ratio,Source Macro F1,Zero-shot Macro F1,Fine-tuned Macro F1,Source Weighted F1,Zero-shot Weighted F1,Fine-tuned Weighted F1
0,EfficientNetB0,99.54 ± 0.41,91.73 ± 1.16,98.93 ± 1.16,7.80 ± 1.46,0.60 ± 1.46,0.9216 ± 0.0144,0.9940 ± 0.0146,0.9885 ± 0.0103,0.9166 ± 0.0122,0.9893 ± 0.0117,0.9955 ± 0.0040,0.9166 ± 0.0122,0.9893 ± 0.0117
1,Lightweight CNN,89.81 ± 2.38,65.87 ± 6.67,87.87 ± 2.54,23.95 ± 7.39,1.95 ± 3.87,0.7342 ± 0.0805,0.9792 ± 0.0415,0.8665 ± 0.0280,0.6517 ± 0.0779,0.8783 ± 0.0256,0.9003 ± 0.0236,0.6517 ± 0.0779,0.8783 ± 0.0256
2,MobileNetV2,98.52 ± 0.74,85.73 ± 3.47,95.07 ± 3.09,12.79 ± 3.69,3.45 ± 3.33,0.8703 ± 0.0370,0.9651 ± 0.0338,0.9642 ± 0.0175,0.8540 ± 0.0354,0.9499 ± 0.0316,0.9854 ± 0.0073,0.8540 ± 0.0354,0.9499 ± 0.0316


In [35]:
paper_table = summary_df[
    [
        "Model",
        "Source Acc (%)",
        "Zero-shot Acc (%)",
        "Fine-tuned Acc (%)",
        "Zero-shot Drop (%)",
        "Fine-tuned Drop (%)",
        "Zero-shot Ratio",
        "Fine-tuned Ratio"
    ]
]

paper_table_path = os.path.join(OUTPUT_DIR, "cross_dataset_transfer_paper_table.csv")
paper_table.to_csv(paper_table_path, index=False)

print("Saved paper table to:", paper_table_path)
paper_table

Saved paper table to: cross_dataset_transfer_20260530-210441/cross_dataset_transfer_paper_table.csv


,Model,Source Acc (%),Zero-shot Acc (%),Fine-tuned Acc (%),Zero-shot Drop (%),Fine-tuned Drop (%),Zero-shot Ratio,Fine-tuned Ratio
0,EfficientNetB0,99.54 ± 0.41,91.73 ± 1.16,98.93 ± 1.16,7.80 ± 1.46,0.60 ± 1.46,0.9216 ± 0.0144,0.9940 ± 0.0146
1,Lightweight CNN,89.81 ± 2.38,65.87 ± 6.67,87.87 ± 2.54,23.95 ± 7.39,1.95 ± 3.87,0.7342 ± 0.0805,0.9792 ± 0.0415
2,MobileNetV2,98.52 ± 0.74,85.73 ± 3.47,95.07 ± 3.09,12.79 ± 3.69,3.45 ± 3.33,0.8703 ± 0.0370,0.9651 ± 0.0338


In [36]:
zip_name = f"{OUTPUT_DIR}.zip"

!zip -r {zip_name} {OUTPUT_DIR}

print("Created:", zip_name)

  adding: cross_dataset_transfer_20260530-210441/ (stored 0%)
  adding: cross_dataset_transfer_20260530-210441/cross_dataset_transfer_per_seed_results.csv (deflated 77%)
  adding: cross_dataset_transfer_20260530-210441/cross_dataset_transfer_paper_table.csv (deflated 46%)
  adding: cross_dataset_transfer_20260530-210441/histories/ (stored 0%)
  adding: cross_dataset_transfer_20260530-210441/histories/MobileNetV2_seed_3407_target_finetune_history.pkl (deflated 56%)
  adding: cross_dataset_transfer_20260530-210441/histories/Lightweight_CNN_seed_2024_target_finetune_history.pkl (deflated 57%)
  adding: cross_dataset_transfer_20260530-210441/histories/MobileNetV2_seed_123_source_history.pkl (deflated 57%)
  adding: cross_dataset_transfer_20260530-210441/histories/Lightweight_CNN_seed_2024_source_history.pkl (deflated 50%)
  adding: cross_dataset_transfer_20260530-210441/histories/MobileNetV2_seed_123_target_finetune_history.pkl (deflated 64%)
  adding: cross_dataset_transfer_20260530-21044

In [38]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [39]:
PREDICTION_FILES = {
    ("Lightweight CNN", "3-class"): "/kaggle/working/path/to/lightweight_cnn_3class_predictions.csv",
    ("Lightweight CNN", "7-class"): "/kaggle/working/path/to/lightweight_cnn_7class_predictions.csv",

    ("MobileNetV2", "3-class"): "/kaggle/working/path/to/mobilenetv2_3class_predictions.csv",
    ("MobileNetV2", "7-class"): "/kaggle/working/path/to/mobilenetv2_7class_predictions.csv",

    ("EfficientNetB0", "3-class"): "/kaggle/working/path/to/efficientnetb0_3class_predictions.csv",
    ("EfficientNetB0", "7-class"): "/kaggle/working/path/to/efficientnetb0_7class_predictions.csv",
}

In [41]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file.endswith(".csv") and ("prediction" in file.lower() or "test_predictions" in file.lower()):
            print(os.path.join(root, file))

In [42]:
for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/kaggle/working/cross_dataset_transfer_20260530-210441/cross_dataset_transfer_per_seed_results.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/cross_dataset_transfer_paper_table.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/cross_dataset_transfer_results.csv


In [40]:
required_columns = [
    "image_path",
    "true_class",
    "pred_class",
    "confidence",
    "correct"
]

prediction_dfs = {}

for (model_name, dataset_name), csv_path in PREDICTION_FILES.items():
    if not os.path.exists(csv_path):
        print(f"Missing file: {csv_path}")
        continue

    df = pd.read_csv(csv_path)

    missing_cols = [col for col in required_columns if col not in df.columns]
    if missing_cols:
        raise ValueError(
            f"{csv_path} is missing columns: {missing_cols}"
        )

    df = df.copy()
    df["confidence"] = df["confidence"].astype(float)

    if df["correct"].dtype != bool:
        df["correct"] = df["correct"].astype(str).str.lower().isin(["true", "1", "yes"])

    prediction_dfs[(model_name, dataset_name)] = df

    print(f"Loaded {model_name} | {dataset_name}: {len(df)} samples")

Missing file: /kaggle/working/path/to/lightweight_cnn_3class_predictions.csv
Missing file: /kaggle/working/path/to/lightweight_cnn_7class_predictions.csv
Missing file: /kaggle/working/path/to/mobilenetv2_3class_predictions.csv
Missing file: /kaggle/working/path/to/mobilenetv2_7class_predictions.csv
Missing file: /kaggle/working/path/to/efficientnetb0_3class_predictions.csv
Missing file: /kaggle/working/path/to/efficientnetb0_7class_predictions.csv


In [43]:
import os

BASE_DIR = "/kaggle/working/cross_dataset_transfer_20260530-210441"

print("All files inside:", BASE_DIR)
for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        print(os.path.join(root, file))

All files inside: /kaggle/working/cross_dataset_transfer_20260530-210441
/kaggle/working/cross_dataset_transfer_20260530-210441/cross_dataset_transfer_per_seed_results.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/cross_dataset_transfer_paper_table.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/cross_dataset_transfer_results.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/histories/MobileNetV2_seed_3407_target_finetune_history.pkl
/kaggle/working/cross_dataset_transfer_20260530-210441/histories/Lightweight_CNN_seed_2024_target_finetune_history.pkl
/kaggle/working/cross_dataset_transfer_20260530-210441/histories/MobileNetV2_seed_123_source_history.pkl
/kaggle/working/cross_dataset_transfer_20260530-210441/histories/Lightweight_CNN_seed_2024_source_history.pkl
/kaggle/working/cross_dataset_transfer_20260530-210441/histories/MobileNetV2_seed_123_target_finetune_history.pkl
/kaggle/working/cross_dataset_transfer_20260530-210441/histories/MobileNetV2_seed

In [45]:
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from sklearn.model_selection import train_test_split

BASE_DIR = "/kaggle/working/cross_dataset_transfer_20260530-210441"

CANONICAL_CLASSES = ["Early Blight", "Late Blight", "Healthy"]
NUM_CLASSES = 3

SEEDS = [42, 123, 2024, 3407, 777]

# Use the same dataset paths from your transfer experiment
SOURCE_DATASET_DIR = "/kaggle/input/datasets/faysalmiah1721758/potato-dataset"
TARGET_DATASET_DIR = "/kaggle/input/datasets/shahadhossin567r7455/potato-leaf-disease-dataset/Potato Leaf DIsease"

CLASS_FOLDER_CANDIDATES = {
    "Early Blight": ["Early Blight", "Early_Blight", "Potato___Early_blight"],
    "Late Blight": ["Late Blight", "Late_Blight", "Potato___Late_blight"],
    "Healthy": ["Healthy", "healthy", "Potato___healthy"]
}

def find_class_folder(dataset_dir, canonical_class):
    folders = [
        folder for folder in os.listdir(dataset_dir)
        if os.path.isdir(os.path.join(dataset_dir, folder))
    ]

    for candidate in CLASS_FOLDER_CANDIDATES[canonical_class]:
        if candidate in folders:
            return os.path.join(dataset_dir, candidate)

    raise ValueError(f"Could not find {canonical_class}. Available folders: {folders}")

def collect_paths_and_labels(dataset_dir):
    valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

    file_paths = []
    labels = []

    for label_index, class_name in enumerate(CANONICAL_CLASSES):
        class_dir = find_class_folder(dataset_dir, class_name)

        for fname in os.listdir(class_dir):
            if fname.lower().endswith(valid_extensions):
                file_paths.append(os.path.join(class_dir, fname))
                labels.append(label_index)

    return np.array(file_paths), np.array(labels)

def make_splits(file_paths, labels, seed):
    X_train, X_temp, y_train, y_temp = train_test_split(
        file_paths,
        labels,
        test_size=0.20,
        stratify=labels,
        random_state=seed
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.50,
        stratify=y_temp,
        random_state=seed
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

def cnn_preprocess(image):
    return image / 255.0

MODEL_SETTINGS = {
    "Lightweight_CNN": {
        "display_name": "Lightweight CNN",
        "image_size": (256, 256),
        "preprocess_fn": cnn_preprocess
    },
    "MobileNetV2": {
        "display_name": "MobileNetV2",
        "image_size": (224, 224),
        "preprocess_fn": mobilenet_preprocess
    },
    "EfficientNetB0": {
        "display_name": "EfficientNetB0",
        "image_size": (224, 224),
        "preprocess_fn": efficientnet_preprocess
    }
}

def load_image_for_prediction(image_path, image_size, preprocess_fn):
    image_raw = tf.io.read_file(image_path)
    image = tf.image.decode_image(image_raw, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])

    image = tf.image.resize(image, image_size)
    image = tf.cast(image, tf.float32)

    image = preprocess_fn(image)
    image = tf.expand_dims(image, axis=0)

    return image

def generate_prediction_csv(model_path, file_paths, labels, model_key, dataset_name, seed, output_dir):
    model = load_model(model_path)

    image_size = MODEL_SETTINGS[model_key]["image_size"]
    preprocess_fn = MODEL_SETTINGS[model_key]["preprocess_fn"]

    records = []

    for image_path, true_label in zip(file_paths, labels):
        image = load_image_for_prediction(image_path, image_size, preprocess_fn)
        probs = model.predict(image, verbose=0)[0]

        pred_label = int(np.argmax(probs))
        confidence = float(np.max(probs))

        records.append({
            "image_path": image_path,
            "true_label": int(true_label),
            "true_class": CANONICAL_CLASSES[int(true_label)],
            "predicted_label": pred_label,
            "pred_class": CANONICAL_CLASSES[pred_label],
            "confidence": confidence,
            "correct": int(true_label) == pred_label,
            "model": MODEL_SETTINGS[model_key]["display_name"],
            "dataset": dataset_name,
            "seed": seed
        })

    pred_df = pd.DataFrame(records)

    os.makedirs(output_dir, exist_ok=True)

    csv_path = os.path.join(
        output_dir,
        f"test_predictions_{model_key}_{dataset_name}_seed_{seed}.csv"
    )

    pred_df.to_csv(csv_path, index=False)

    print("Saved:", csv_path)

    return csv_path

In [46]:
source_paths, source_labels = collect_paths_and_labels(SOURCE_DATASET_DIR)
target_paths, target_labels = collect_paths_and_labels(TARGET_DATASET_DIR)

prediction_output_dir = os.path.join(BASE_DIR, "prediction_csvs")
generated_prediction_files = []

for model_key in ["Lightweight_CNN", "MobileNetV2", "EfficientNetB0"]:
    for seed in SEEDS:
        _, _, source_test, _, _, y_source_test = make_splits(
            source_paths,
            source_labels,
            seed
        )

        _, _, target_test, _, _, y_target_test = make_splits(
            target_paths,
            target_labels,
            seed
        )

        source_model_path = os.path.join(
            BASE_DIR,
            "models",
            f"{model_key}_seed_{seed}_source.keras"
        )

        target_model_path = os.path.join(
            BASE_DIR,
            "models",
            f"{model_key}_seed_{seed}_target_finetuned.keras"
        )

        if os.path.exists(source_model_path):
            generated_prediction_files.append(
                generate_prediction_csv(
                    model_path=source_model_path,
                    file_paths=source_test,
                    labels=y_source_test,
                    model_key=model_key,
                    dataset_name="source_3class",
                    seed=seed,
                    output_dir=prediction_output_dir
                )
            )

            generated_prediction_files.append(
                generate_prediction_csv(
                    model_path=source_model_path,
                    file_paths=target_test,
                    labels=y_target_test,
                    model_key=model_key,
                    dataset_name="zero_shot_target_3class",
                    seed=seed,
                    output_dir=prediction_output_dir
                )
            )

        if os.path.exists(target_model_path):
            generated_prediction_files.append(
                generate_prediction_csv(
                    model_path=target_model_path,
                    file_paths=target_test,
                    labels=y_target_test,
                    model_key=model_key,
                    dataset_name="finetuned_target_3class",
                    seed=seed,
                    output_dir=prediction_output_dir
                )
            )

print("Generated prediction files:")
for path in generated_prediction_files:
    print(path)

Saved: /kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_Lightweight_CNN_source_3class_seed_42.csv
Saved: /kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_Lightweight_CNN_zero_shot_target_3class_seed_42.csv
Saved: /kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_Lightweight_CNN_finetuned_target_3class_seed_42.csv
Saved: /kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_Lightweight_CNN_source_3class_seed_123.csv
Saved: /kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_Lightweight_CNN_zero_shot_target_3class_seed_123.csv
Saved: /kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_Lightweight_CNN_finetuned_target_3class_seed_123.csv
Saved: /kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_Lightweight_CNN_source_3class_seed_2024.

In [47]:
for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file.endswith(".csv") and "test_predictions" in file:
            print(os.path.join(root, file))

/kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_MobileNetV2_finetuned_target_3class_seed_42.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_MobileNetV2_source_3class_seed_777.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_MobileNetV2_source_3class_seed_3407.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_Lightweight_CNN_zero_shot_target_3class_seed_777.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_EfficientNetB0_source_3class_seed_777.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_EfficientNetB0_zero_shot_target_3class_seed_2024.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_Lightweight_CNN_finetuned_target_3class_seed_42.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/pr

In [48]:
prediction_files = []

for root, dirs, files in os.walk("/kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs"):
    for file in files:
        if file.endswith(".csv") and file.startswith("test_predictions"):
            prediction_files.append(os.path.join(root, file))

print("Found prediction files:", len(prediction_files))

for path in prediction_files:
    print(path)

Found prediction files: 45
/kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_MobileNetV2_finetuned_target_3class_seed_42.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_MobileNetV2_source_3class_seed_777.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_MobileNetV2_source_3class_seed_3407.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_Lightweight_CNN_zero_shot_target_3class_seed_777.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_EfficientNetB0_source_3class_seed_777.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_EfficientNetB0_zero_shot_target_3class_seed_2024.csv
/kaggle/working/cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_Lightweight_CNN_finetuned_target_3class_seed_42.csv
/kaggle/working/cross_dataset_

In [49]:
required_columns = [
    "image_path",
    "true_class",
    "pred_class",
    "confidence",
    "correct",
    "model",
    "dataset",
    "seed"
]

prediction_dfs = []

for csv_path in prediction_files:
    df = pd.read_csv(csv_path)

    missing_cols = [col for col in required_columns if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{csv_path} missing columns: {missing_cols}")

    df["confidence"] = df["confidence"].astype(float)

    if df["correct"].dtype != bool:
        df["correct"] = df["correct"].astype(str).str.lower().isin(["true", "1", "yes"])

    prediction_dfs.append(df)

all_predictions_df = pd.concat(prediction_dfs, ignore_index=True)

print("Total prediction rows:", len(all_predictions_df))
display(all_predictions_df.head())

Total prediction rows: 7740


,image_path,true_label,true_class,predicted_label,pred_class,confidence,correct,model,dataset,seed
0,/kaggle/input/datasets/shahadhossin567r7455/po...,1,Late Blight,1,Late Blight,0.840941,True,MobileNetV2,finetuned_target_3class,42
1,/kaggle/input/datasets/shahadhossin567r7455/po...,0,Early Blight,0,Early Blight,0.999999,True,MobileNetV2,finetuned_target_3class,42
2,/kaggle/input/datasets/shahadhossin567r7455/po...,2,Healthy,2,Healthy,0.995557,True,MobileNetV2,finetuned_target_3class,42
3,/kaggle/input/datasets/shahadhossin567r7455/po...,2,Healthy,2,Healthy,0.999999,True,MobileNetV2,finetuned_target_3class,42
4,/kaggle/input/datasets/shahadhossin567r7455/po...,2,Healthy,2,Healthy,0.998354,True,MobileNetV2,finetuned_target_3class,42


In [50]:
def compute_decision_metrics(df, scheme_name, alpha=None, beta=None):
    total_samples = len(df)
    total_wrong = (~df["correct"]).sum()

    accept_df = df[df["decision_region"] == "Accept"]
    defer_df = df[df["decision_region"] == "Defer"]
    reject_df = df[df["decision_region"] == "Reject"]

    accept_count = len(accept_df)
    defer_count = len(defer_df)
    reject_count = len(reject_df)

    accept_percentage = accept_count / total_samples * 100
    defer_percentage = defer_count / total_samples * 100
    reject_percentage = reject_count / total_samples * 100

    if accept_count > 0:
        accept_accuracy = accept_df["correct"].mean() * 100
    else:
        accept_accuracy = np.nan

    captured_wrong = df[
        (~df["correct"]) &
        (df["decision_region"].isin(["Defer", "Reject"]))
    ]

    if total_wrong > 0:
        wrong_capture_rate = len(captured_wrong) / total_wrong * 100
    else:
        wrong_capture_rate = np.nan

    return {
        "scheme": scheme_name,
        "alpha": alpha,
        "beta": beta,
        "total_samples": total_samples,
        "accept_count": accept_count,
        "defer_count": defer_count,
        "reject_count": reject_count,
        "accept_percentage": accept_percentage,
        "defer_percentage": defer_percentage,
        "reject_percentage": reject_percentage,
        "accept_accuracy": accept_accuracy,
        "wrong_capture_rate": wrong_capture_rate,
        "total_wrong": int(total_wrong),
        "captured_wrong": int(len(captured_wrong))
    }

In [51]:
two_way_alphas = [0.85, 0.90, 0.95]

three_way_thresholds = [
    (0.95, 0.70),
    (0.90, 0.60),
    (0.85, 0.50)
]

ablation_rows = []

group_cols = ["model", "dataset", "seed"]

for (model_name, dataset_name, seed), group_df in all_predictions_df.groupby(group_cols):
    group_df = group_df.copy()

    # A. Standard hard prediction
    standard_df = group_df.copy()
    standard_df["decision_region"] = "Accept"

    result = compute_decision_metrics(
        standard_df,
        scheme_name="Standard",
        alpha=None,
        beta=None
    )
    result["model"] = model_name
    result["dataset"] = dataset_name
    result["seed"] = seed
    ablation_rows.append(result)

    # B. Two-way decision
    for alpha in two_way_alphas:
        two_way_df = group_df.copy()
        two_way_df["decision_region"] = np.where(
            two_way_df["confidence"] >= alpha,
            "Accept",
            "Reject"
        )

        result = compute_decision_metrics(
            two_way_df,
            scheme_name="Two-way",
            alpha=alpha,
            beta=None
        )
        result["model"] = model_name
        result["dataset"] = dataset_name
        result["seed"] = seed
        ablation_rows.append(result)

    # C. Three-way decision
    for alpha, beta in three_way_thresholds:
        three_way_df = group_df.copy()

        conditions = [
            three_way_df["confidence"] >= alpha,
            (three_way_df["confidence"] >= beta) & (three_way_df["confidence"] < alpha),
            three_way_df["confidence"] < beta
        ]

        choices = ["Accept", "Defer", "Reject"]

        three_way_df["decision_region"] = np.select(
            conditions,
            choices,
            default="Reject"
        )

        result = compute_decision_metrics(
            three_way_df,
            scheme_name="Three-way",
            alpha=alpha,
            beta=beta
        )
        result["model"] = model_name
        result["dataset"] = dataset_name
        result["seed"] = seed
        ablation_rows.append(result)

ablation_df = pd.DataFrame(ablation_rows)

ablation_df = ablation_df[
    [
        "model",
        "dataset",
        "seed",
        "scheme",
        "alpha",
        "beta",
        "total_samples",
        "accept_count",
        "defer_count",
        "reject_count",
        "accept_percentage",
        "defer_percentage",
        "reject_percentage",
        "accept_accuracy",
        "wrong_capture_rate",
        "total_wrong",
        "captured_wrong"
    ]
]

ablation_results_path = "ablation_decision_mechanism_results.csv"
ablation_df.to_csv(ablation_results_path, index=False)

print("Saved:", ablation_results_path)
display(ablation_df.head())

Saved: ablation_decision_mechanism_results.csv


,model,dataset,seed,scheme,alpha,beta,total_samples,accept_count,defer_count,reject_count,accept_percentage,defer_percentage,reject_percentage,accept_accuracy,wrong_capture_rate,total_wrong,captured_wrong
0,EfficientNetB0,finetuned_target_3class,42,Standard,NaN,NaN,150,150,0,0,100.000000,0.0,0.000000,96.666667,0.0,5,0
1,EfficientNetB0,finetuned_target_3class,42,Two-way,0.85,NaN,150,145,0,5,96.666667,0.0,3.333333,99.310345,80.0,5,4
2,EfficientNetB0,finetuned_target_3class,42,Two-way,0.90,NaN,150,144,0,6,96.000000,0.0,4.000000,99.305556,80.0,5,4
3,EfficientNetB0,finetuned_target_3class,42,Two-way,0.95,NaN,150,140,0,10,93.333333,0.0,6.666667,100.000000,100.0,5,5
4,EfficientNetB0,finetuned_target_3class,42,Three-way,0.95,0.7,150,140,6,4,93.333333,4.0,2.666667,100.000000,100.0,5,5


In [52]:
summary_metrics = [
    "accept_percentage",
    "defer_percentage",
    "reject_percentage",
    "accept_accuracy",
    "wrong_capture_rate"
]

summary_rows = []

for keys, group in ablation_df.groupby(["model", "dataset", "scheme", "alpha", "beta"], dropna=False):
    model_name, dataset_name, scheme, alpha, beta = keys

    row = {
        "model": model_name,
        "dataset": dataset_name,
        "scheme": scheme,
        "alpha": alpha,
        "beta": beta
    }

    for metric in summary_metrics:
        row[f"{metric}_mean"] = group[metric].mean()
        row[f"{metric}_std"] = group[metric].std(ddof=0)

    summary_rows.append(row)

ablation_summary_df = pd.DataFrame(summary_rows)

ablation_summary_path = "ablation_decision_mechanism_summary_mean_std.csv"
ablation_summary_df.to_csv(ablation_summary_path, index=False)

print("Saved:", ablation_summary_path)
display(ablation_summary_df.head())

Saved: ablation_decision_mechanism_summary_mean_std.csv


,model,dataset,scheme,alpha,beta,accept_percentage_mean,accept_percentage_std,defer_percentage_mean,defer_percentage_std,reject_percentage_mean,reject_percentage_std,accept_accuracy_mean,accept_accuracy_std,wrong_capture_rate_mean,wrong_capture_rate_std
0,EfficientNetB0,finetuned_target_3class,Standard,NaN,NaN,100.000000,0.000000,0.000000,0.000000,0.000000,0.000000,98.933333,1.162373,0.0,0.000000
1,EfficientNetB0,finetuned_target_3class,Three-way,0.85,0.5,96.800000,0.498888,3.066667,0.533333,0.133333,0.266667,99.862069,0.275862,95.0,8.660254
2,EfficientNetB0,finetuned_target_3class,Three-way,0.90,0.6,95.866667,1.066667,3.200000,1.146977,0.933333,0.679869,99.861111,0.277778,95.0,8.660254
3,EfficientNetB0,finetuned_target_3class,Three-way,0.95,0.7,93.466667,1.707500,5.066667,1.717880,1.466667,0.653197,100.000000,0.000000,100.0,0.000000
4,EfficientNetB0,finetuned_target_3class,Two-way,0.85,NaN,96.800000,0.498888,0.000000,0.000000,3.200000,0.498888,99.862069,0.275862,95.0,8.660254


In [53]:
table_a = ablation_summary_df[
    (
        (ablation_summary_df["scheme"] == "Standard") |
        (
            (ablation_summary_df["scheme"] == "Two-way") &
            (ablation_summary_df["alpha"] == 0.90)
        ) |
        (
            (ablation_summary_df["scheme"] == "Three-way") &
            (ablation_summary_df["alpha"] == 0.90) &
            (ablation_summary_df["beta"] == 0.60)
        )
    )
].copy()

def format_mean_std(row, metric):
    return f"{row[f'{metric}_mean']:.2f} ± {row[f'{metric}_std']:.2f}"

table_a["setting"] = table_a.apply(
    lambda row: (
        "Standard"
        if row["scheme"] == "Standard"
        else "Two-way α=0.90"
        if row["scheme"] == "Two-way"
        else "Three-way α=0.90, β=0.60"
    ),
    axis=1
)

table_a_formatted = pd.DataFrame({
    "Model": table_a["model"],
    "Dataset": table_a["dataset"],
    "Setting": table_a["setting"],
    "Accept (%)": table_a.apply(lambda row: format_mean_std(row, "accept_percentage"), axis=1),
    "Defer (%)": table_a.apply(lambda row: format_mean_std(row, "defer_percentage"), axis=1),
    "Reject (%)": table_a.apply(lambda row: format_mean_std(row, "reject_percentage"), axis=1),
    "Accept Accuracy (%)": table_a.apply(lambda row: format_mean_std(row, "accept_accuracy"), axis=1),
    "Wrong Capture Rate (%)": table_a.apply(lambda row: format_mean_std(row, "wrong_capture_rate"), axis=1)
})

table_a_formatted = table_a_formatted.sort_values(["Dataset", "Model", "Setting"])

table_a_path = "ablation_table_A_standard_two_way_three_way.csv"
table_a_formatted.to_csv(table_a_path, index=False)

print("Saved:", table_a_path)
display(table_a_formatted)

Saved: ablation_table_A_standard_two_way_three_way.csv


,Model,Dataset,Setting,Accept (%),Defer (%),Reject (%),Accept Accuracy (%),Wrong Capture Rate (%)
0,EfficientNetB0,finetuned_target_3class,Standard,100.00 ± 0.00,0.00 ± 0.00,0.00 ± 0.00,98.93 ± 1.16,0.00 ± 0.00
2,EfficientNetB0,finetuned_target_3class,"Three-way α=0.90, β=0.60",95.87 ± 1.07,3.20 ± 1.15,0.93 ± 0.68,99.86 ± 0.28,95.00 ± 8.66
5,EfficientNetB0,finetuned_target_3class,Two-way α=0.90,95.87 ± 1.07,0.00 ± 0.00,4.13 ± 1.07,99.86 ± 0.28,95.00 ± 8.66
21,Lightweight CNN,finetuned_target_3class,Standard,100.00 ± 0.00,0.00 ± 0.00,0.00 ± 0.00,87.87 ± 2.54,0.00 ± 0.00
23,Lightweight CNN,finetuned_target_3class,"Three-way α=0.90, β=0.60",51.47 ± 3.54,37.60 ± 2.00,10.93 ± 2.69,98.49 ± 1.46,93.64 ± 6.05
26,Lightweight CNN,finetuned_target_3class,Two-way α=0.90,51.47 ± 3.54,0.00 ± 0.00,48.53 ± 3.54,98.49 ± 1.46,93.64 ± 6.05
42,MobileNetV2,finetuned_target_3class,Standard,100.00 ± 0.00,0.00 ± 0.00,0.00 ± 0.00,95.07 ± 3.09,0.00 ± 0.00
44,MobileNetV2,finetuned_target_3class,"Three-way α=0.90, β=0.60",91.87 ± 3.80,6.53 ± 3.80,1.60 ± 1.08,97.77 ± 1.95,68.47 ± 20.80
47,MobileNetV2,finetuned_target_3class,Two-way α=0.90,91.87 ± 3.80,0.00 ± 0.00,8.13 ± 3.80,97.77 ± 1.95,68.47 ± 20.80
7,EfficientNetB0,source_3class,Standard,100.00 ± 0.00,0.00 ± 0.00,0.00 ± 0.00,99.54 ± 0.41,0.00 ± 0.00


In [54]:
table_b = ablation_summary_df[
    ablation_summary_df["scheme"] == "Three-way"
].copy()

table_b["threshold_pair"] = table_b.apply(
    lambda row: f"α={row['alpha']:.2f}, β={row['beta']:.2f}",
    axis=1
)

table_b_formatted = pd.DataFrame({
    "Model": table_b["model"],
    "Dataset": table_b["dataset"],
    "Threshold Pair": table_b["threshold_pair"],
    "Accept (%)": table_b.apply(lambda row: format_mean_std(row, "accept_percentage"), axis=1),
    "Defer (%)": table_b.apply(lambda row: format_mean_std(row, "defer_percentage"), axis=1),
    "Reject (%)": table_b.apply(lambda row: format_mean_std(row, "reject_percentage"), axis=1),
    "Accept Accuracy (%)": table_b.apply(lambda row: format_mean_std(row, "accept_accuracy"), axis=1),
    "Wrong Capture Rate (%)": table_b.apply(lambda row: format_mean_std(row, "wrong_capture_rate"), axis=1)
})

table_b_formatted = table_b_formatted.sort_values(["Dataset", "Model", "Threshold Pair"])

threshold_results_path = "ablation_threshold_sensitivity_results.csv"
table_b_formatted.to_csv(threshold_results_path, index=False)

print("Saved:", threshold_results_path)
display(table_b_formatted)

Saved: ablation_threshold_sensitivity_results.csv


,Model,Dataset,Threshold Pair,Accept (%),Defer (%),Reject (%),Accept Accuracy (%),Wrong Capture Rate (%)
1,EfficientNetB0,finetuned_target_3class,"α=0.85, β=0.50",96.80 ± 0.50,3.07 ± 0.53,0.13 ± 0.27,99.86 ± 0.28,95.00 ± 8.66
2,EfficientNetB0,finetuned_target_3class,"α=0.90, β=0.60",95.87 ± 1.07,3.20 ± 1.15,0.93 ± 0.68,99.86 ± 0.28,95.00 ± 8.66
3,EfficientNetB0,finetuned_target_3class,"α=0.95, β=0.70",93.47 ± 1.71,5.07 ± 1.72,1.47 ± 0.65,100.00 ± 0.00,100.00 ± 0.00
22,Lightweight CNN,finetuned_target_3class,"α=0.85, β=0.50",61.33 ± 2.76,35.33 ± 2.39,3.33 ± 0.42,97.41 ± 1.57,87.49 ± 5.66
23,Lightweight CNN,finetuned_target_3class,"α=0.90, β=0.60",51.47 ± 3.54,37.60 ± 2.00,10.93 ± 2.69,98.49 ± 1.46,93.64 ± 6.05
24,Lightweight CNN,finetuned_target_3class,"α=0.95, β=0.70",32.00 ± 6.75,49.33 ± 4.64,18.67 ± 3.18,98.87 ± 1.42,96.90 ± 3.81
43,MobileNetV2,finetuned_target_3class,"α=0.85, β=0.50",92.53 ± 4.07,7.20 ± 4.22,0.27 ± 0.33,97.49 ± 1.92,64.02 ± 20.71
44,MobileNetV2,finetuned_target_3class,"α=0.90, β=0.60",91.87 ± 3.80,6.53 ± 3.80,1.60 ± 1.08,97.77 ± 1.95,68.47 ± 20.80
45,MobileNetV2,finetuned_target_3class,"α=0.95, β=0.70",88.53 ± 4.80,7.60 ± 3.54,3.87 ± 1.60,98.27 ± 1.30,75.46 ± 13.14
8,EfficientNetB0,source_3class,"α=0.85, β=0.50",96.30 ± 0.72,3.70 ± 0.72,0.00 ± 0.00,99.81 ± 0.38,66.67 ± 47.14


In [55]:
!zip -r ablation_decision_mechanism_outputs.zip \
ablation_decision_mechanism_results.csv \
ablation_decision_mechanism_summary_mean_std.csv \
ablation_table_A_standard_two_way_three_way.csv \
ablation_threshold_sensitivity_results.csv

print("Created: ablation_decision_mechanism_outputs.zip")

  adding: ablation_decision_mechanism_results.csv (deflated 87%)
  adding: ablation_decision_mechanism_summary_mean_std.csv (deflated 77%)
  adding: ablation_table_A_standard_two_way_three_way.csv (deflated 77%)
  adding: ablation_threshold_sensitivity_results.csv (deflated 70%)
Created: ablation_decision_mechanism_outputs.zip


In [56]:
table_b_main = table_b_formatted[
    table_b_formatted["Dataset"].str.contains("7", case=False, na=False)
].copy()

table_b_main_path = "ablation_threshold_sensitivity_7class_main_paper.csv"
table_b_main.to_csv(table_b_main_path, index=False)

table_b_main

,Model,Dataset,Threshold Pair,Accept (%),Defer (%),Reject (%),Accept Accuracy (%),Wrong Capture Rate (%)


In [57]:
def region_accuracy(df, region):
    region_df = df[df["decision_region"] == region]

    if len(region_df) == 0:
        return np.nan

    return region_df["correct"].mean() * 100


def compute_extended_decision_metrics(df, scheme_name, alpha=None, beta=None):
    total_samples = len(df)
    total_wrong = (~df["correct"]).sum()

    accept_df = df[df["decision_region"] == "Accept"]
    defer_df = df[df["decision_region"] == "Defer"]
    reject_df = df[df["decision_region"] == "Reject"]

    accept_percentage = len(accept_df) / total_samples * 100
    defer_percentage = len(defer_df) / total_samples * 100
    reject_percentage = len(reject_df) / total_samples * 100

    accept_accuracy = region_accuracy(df, "Accept")
    defer_accuracy = region_accuracy(df, "Defer")
    reject_accuracy = region_accuracy(df, "Reject")

    captured_wrong = df[
        (~df["correct"]) &
        (df["decision_region"].isin(["Defer", "Reject"]))
    ]

    if total_wrong > 0:
        wrong_capture_rate = len(captured_wrong) / total_wrong * 100
    else:
        wrong_capture_rate = np.nan

    return {
        "scheme": scheme_name,
        "alpha": alpha,
        "beta": beta,
        "accept_percentage": accept_percentage,
        "accept_accuracy": accept_accuracy,
        "defer_percentage": defer_percentage,
        "defer_accuracy": defer_accuracy,
        "reject_percentage": reject_percentage,
        "reject_accuracy": reject_accuracy,
        "wrong_capture_rate": wrong_capture_rate
    }

In [58]:
two_way_alphas = [0.85, 0.90, 0.95]

three_way_thresholds = [
    (0.95, 0.70),
    (0.90, 0.60),
    (0.85, 0.50)
]

extended_rows = []

group_cols = ["model", "dataset", "seed"]

for (model_name, dataset_name, seed), group_df in all_predictions_df.groupby(group_cols):
    group_df = group_df.copy()

    # Standard hard prediction
    standard_df = group_df.copy()
    standard_df["decision_region"] = "Accept"

    result = compute_extended_decision_metrics(
        standard_df,
        scheme_name="Standard",
        alpha=None,
        beta=None
    )
    result["model"] = model_name
    result["dataset"] = dataset_name
    result["seed"] = seed
    extended_rows.append(result)

    # Two-way decision
    for alpha in two_way_alphas:
        two_way_df = group_df.copy()
        two_way_df["decision_region"] = np.where(
            two_way_df["confidence"] >= alpha,
            "Accept",
            "Reject"
        )

        result = compute_extended_decision_metrics(
            two_way_df,
            scheme_name="Two-way",
            alpha=alpha,
            beta=None
        )
        result["model"] = model_name
        result["dataset"] = dataset_name
        result["seed"] = seed
        extended_rows.append(result)

    # Three-way decision
    for alpha, beta in three_way_thresholds:
        three_way_df = group_df.copy()

        conditions = [
            three_way_df["confidence"] >= alpha,
            (three_way_df["confidence"] >= beta) & (three_way_df["confidence"] < alpha),
            three_way_df["confidence"] < beta
        ]

        choices = ["Accept", "Defer", "Reject"]

        three_way_df["decision_region"] = np.select(
            conditions,
            choices,
            default="Reject"
        )

        result = compute_extended_decision_metrics(
            three_way_df,
            scheme_name="Three-way",
            alpha=alpha,
            beta=beta
        )
        result["model"] = model_name
        result["dataset"] = dataset_name
        result["seed"] = seed
        extended_rows.append(result)

extended_ablation_df = pd.DataFrame(extended_rows)

extended_ablation_path = "ablation_extended_region_metrics_per_seed.csv"
extended_ablation_df.to_csv(extended_ablation_path, index=False)

print("Saved:", extended_ablation_path)
extended_ablation_df.head()

Saved: ablation_extended_region_metrics_per_seed.csv


,scheme,alpha,beta,accept_percentage,accept_accuracy,defer_percentage,defer_accuracy,reject_percentage,reject_accuracy,wrong_capture_rate,model,dataset,seed
0,Standard,NaN,NaN,100.000000,96.666667,0.0,NaN,0.000000,NaN,0.0,EfficientNetB0,finetuned_target_3class,42
1,Two-way,0.85,NaN,96.666667,99.310345,0.0,NaN,3.333333,20.000000,80.0,EfficientNetB0,finetuned_target_3class,42
2,Two-way,0.90,NaN,96.000000,99.305556,0.0,NaN,4.000000,33.333333,80.0,EfficientNetB0,finetuned_target_3class,42
3,Two-way,0.95,NaN,93.333333,100.000000,0.0,NaN,6.666667,50.000000,100.0,EfficientNetB0,finetuned_target_3class,42
4,Three-way,0.95,0.7,93.333333,100.000000,4.0,83.333333,2.666667,0.000000,100.0,EfficientNetB0,finetuned_target_3class,42


In [60]:
main_table_df = extended_ablation_df[
    (
        (extended_ablation_df["scheme"] == "Standard") |
        (
            (extended_ablation_df["scheme"] == "Two-way") &
            (extended_ablation_df["alpha"] == 0.90)
        ) |
        (
            (extended_ablation_df["scheme"] == "Three-way") &
            (extended_ablation_df["alpha"] == 0.90) &
            (extended_ablation_df["beta"] == 0.60)
        )
    )
].copy()

In [61]:
summary_metrics = [
    "accept_percentage",
    "accept_accuracy",
    "defer_percentage",
    "defer_accuracy",
    "reject_percentage",
    "reject_accuracy",
    "wrong_capture_rate"
]

summary_rows = []

for keys, group in main_table_df.groupby(["model", "dataset", "scheme", "alpha", "beta"], dropna=False):
    model_name, dataset_name, scheme, alpha, beta = keys

    row = {
        "model": model_name,
        "dataset": dataset_name,
        "scheme": scheme,
        "alpha": alpha,
        "beta": beta
    }

    for metric in summary_metrics:
        row[f"{metric}_mean"] = group[metric].mean()
        row[f"{metric}_std"] = group[metric].std(ddof=0)

    summary_rows.append(row)

main_summary_df = pd.DataFrame(summary_rows)

In [62]:
def fmt_mean_std(row, metric):
    mean_value = row[f"{metric}_mean"]
    std_value = row[f"{metric}_std"]

    if pd.isna(mean_value):
        return "-"

    return f"{mean_value:.2f} ± {std_value:.2f}"


def decision_label(row):
    if row["scheme"] == "Standard":
        return "Standard"
    elif row["scheme"] == "Two-way":
        return "Two-way α=0.90"
    else:
        return "Three-way α=0.90, β=0.60"


paper_extended_table = pd.DataFrame({
    "Model": main_summary_df["model"],
    "Dataset": main_summary_df["dataset"],
    "Decision": main_summary_df.apply(decision_label, axis=1),
    "Accept %": main_summary_df.apply(lambda row: fmt_mean_std(row, "accept_percentage"), axis=1),
    "Accept Acc.": main_summary_df.apply(lambda row: fmt_mean_std(row, "accept_accuracy"), axis=1),
    "Defer %": main_summary_df.apply(lambda row: fmt_mean_std(row, "defer_percentage"), axis=1),
    "Defer Acc.": main_summary_df.apply(lambda row: fmt_mean_std(row, "defer_accuracy"), axis=1),
    "Reject %": main_summary_df.apply(lambda row: fmt_mean_std(row, "reject_percentage"), axis=1),
    "Reject Acc.": main_summary_df.apply(lambda row: fmt_mean_std(row, "reject_accuracy"), axis=1),
    "WCR": main_summary_df.apply(lambda row: fmt_mean_std(row, "wrong_capture_rate"), axis=1)
})

paper_extended_table = paper_extended_table.sort_values(["Dataset", "Model", "Decision"])

paper_extended_table_path = "ablation_main_paper_extended_region_table.csv"
paper_extended_table.to_csv(paper_extended_table_path, index=False)

print("Saved:", paper_extended_table_path)
paper_extended_table

Saved: ablation_main_paper_extended_region_table.csv


,Model,Dataset,Decision,Accept %,Accept Acc.,Defer %,Defer Acc.,Reject %,Reject Acc.,WCR
0,EfficientNetB0,finetuned_target_3class,Standard,100.00 ± 0.00,98.93 ± 1.16,0.00 ± 0.00,-,0.00 ± 0.00,-,0.00 ± 0.00
1,EfficientNetB0,finetuned_target_3class,"Three-way α=0.90, β=0.60",95.87 ± 1.07,99.86 ± 0.28,3.20 ± 1.15,85.83 ± 13.33,0.93 ± 0.68,62.50 ± 41.46,95.00 ± 8.66
2,EfficientNetB0,finetuned_target_3class,Two-way α=0.90,95.87 ± 1.07,99.86 ± 0.28,0.00 ± 0.00,-,4.13 ± 1.07,76.11 ± 22.88,95.00 ± 8.66
9,Lightweight CNN,finetuned_target_3class,Standard,100.00 ± 0.00,87.87 ± 2.54,0.00 ± 0.00,-,0.00 ± 0.00,-,0.00 ± 0.00
10,Lightweight CNN,finetuned_target_3class,"Three-way α=0.90, β=0.60",51.47 ± 3.54,98.49 ± 1.46,37.60 ± 2.00,83.03 ± 3.75,10.93 ± 2.69,54.66 ± 15.93,93.64 ± 6.05
11,Lightweight CNN,finetuned_target_3class,Two-way α=0.90,51.47 ± 3.54,98.49 ± 1.46,0.00 ± 0.00,-,48.53 ± 3.54,76.56 ± 4.94,93.64 ± 6.05
18,MobileNetV2,finetuned_target_3class,Standard,100.00 ± 0.00,95.07 ± 3.09,0.00 ± 0.00,-,0.00 ± 0.00,-,0.00 ± 0.00
19,MobileNetV2,finetuned_target_3class,"Three-way α=0.90, β=0.60",91.87 ± 3.80,97.77 ± 1.95,6.53 ± 3.80,73.88 ± 18.77,1.60 ± 1.08,51.67 ± 9.57,68.47 ± 20.80
20,MobileNetV2,finetuned_target_3class,Two-way α=0.90,91.87 ± 3.80,97.77 ± 1.95,0.00 ± 0.00,-,8.13 ± 3.80,66.76 ± 12.14,68.47 ± 20.80
3,EfficientNetB0,source_3class,Standard,100.00 ± 0.00,99.54 ± 0.41,0.00 ± 0.00,-,0.00 ± 0.00,-,0.00 ± 0.00


In [63]:
gap_source_df = extended_ablation_df[
    (extended_ablation_df["scheme"] == "Three-way") &
    (extended_ablation_df["alpha"] == 0.90) &
    (extended_ablation_df["beta"] == 0.60)
].copy()

gap_metrics = [
    "accept_accuracy",
    "defer_accuracy",
    "reject_accuracy"
]

gap_summary_rows = []

for keys, group in gap_source_df.groupby(["model", "dataset"]):
    model_name, dataset_name = keys

    accept_mean = group["accept_accuracy"].mean()
    accept_std = group["accept_accuracy"].std(ddof=0)

    defer_mean = group["defer_accuracy"].mean()
    defer_std = group["defer_accuracy"].std(ddof=0)

    reject_mean = group["reject_accuracy"].mean()
    reject_std = group["reject_accuracy"].std(ddof=0)

    gap_ad = accept_mean - defer_mean
    gap_dr = defer_mean - reject_mean

    gap_summary_rows.append({
        "Model": model_name,
        "Dataset": dataset_name,
        "Accept Acc. Mean": accept_mean,
        "Accept Acc. Std": accept_std,
        "Defer Acc. Mean": defer_mean,
        "Defer Acc. Std": defer_std,
        "Reject Acc. Mean": reject_mean,
        "Reject Acc. Std": reject_std,
        "Gap(A,D)": gap_ad,
        "Gap(D,R)": gap_dr
    })

gap_table_raw = pd.DataFrame(gap_summary_rows)

In [64]:
def fmt_acc(mean_value, std_value):
    if pd.isna(mean_value):
        return "-"

    return f"{mean_value:.2f} ± {std_value:.2f}"


def fmt_gap(value):
    if pd.isna(value):
        return "-"

    return f"{value:.2f}"


gap_table = pd.DataFrame({
    "Model": gap_table_raw["Model"],
    "Dataset": gap_table_raw["Dataset"],
    "Accept Acc.": gap_table_raw.apply(
        lambda row: fmt_acc(row["Accept Acc. Mean"], row["Accept Acc. Std"]),
        axis=1
    ),
    "Defer Acc.": gap_table_raw.apply(
        lambda row: fmt_acc(row["Defer Acc. Mean"], row["Defer Acc. Std"]),
        axis=1
    ),
    "Reject Acc.": gap_table_raw.apply(
        lambda row: fmt_acc(row["Reject Acc. Mean"], row["Reject Acc. Std"]),
        axis=1
    ),
    "Gap(A,D)": gap_table_raw["Gap(A,D)"].apply(fmt_gap),
    "Gap(D,R)": gap_table_raw["Gap(D,R)"].apply(fmt_gap)
})

gap_table = gap_table.sort_values(["Dataset", "Model"])

gap_table_path = "ablation_three_way_region_accuracy_gap_table.csv"
gap_table.to_csv(gap_table_path, index=False)

print("Saved:", gap_table_path)
gap_table

Saved: ablation_three_way_region_accuracy_gap_table.csv


,Model,Dataset,Accept Acc.,Defer Acc.,Reject Acc.,"Gap(A,D)","Gap(D,R)"
0,EfficientNetB0,finetuned_target_3class,99.86 ± 0.28,85.83 ± 13.33,62.50 ± 41.46,14.03,23.33
3,Lightweight CNN,finetuned_target_3class,98.49 ± 1.46,83.03 ± 3.75,54.66 ± 15.93,15.45,28.37
6,MobileNetV2,finetuned_target_3class,97.77 ± 1.95,73.88 ± 18.77,51.67 ± 9.57,23.89,22.21
1,EfficientNetB0,source_3class,99.90 ± 0.20,93.48 ± 5.67,90.00 ± 20.00,6.43,3.48
4,Lightweight CNN,source_3class,99.00 ± 0.90,86.35 ± 5.28,53.59 ± 9.99,12.64,32.76
7,MobileNetV2,source_3class,99.51 ± 0.31,81.11 ± 12.25,60.00 ± 37.42,18.40,21.11
2,EfficientNetB0,zero_shot_target_3class,99.82 ± 0.36,79.88 ± 7.46,49.91 ± 14.02,19.94,29.97
5,Lightweight CNN,zero_shot_target_3class,78.64 ± 11.15,58.89 ± 7.11,57.92 ± 6.29,19.74,0.97
8,MobileNetV2,zero_shot_target_3class,96.72 ± 1.03,53.53 ± 16.27,34.59 ± 7.50,43.19,18.94


In [65]:
for _, row in gap_table_raw.sort_values(["Dataset", "Model"]).iterrows():
    print(
        f"{row['Model']} | {row['Dataset']}: "
        f"Gap(A,D) = {row['Accept Acc. Mean']:.2f} - {row['Defer Acc. Mean']:.2f} = {row['Gap(A,D)']:.2f}; "
        f"Gap(D,R) = {row['Defer Acc. Mean']:.2f} - {row['Reject Acc. Mean']:.2f} = {row['Gap(D,R)']:.2f}"
    )

EfficientNetB0 | finetuned_target_3class: Gap(A,D) = 99.86 - 85.83 = 14.03; Gap(D,R) = 85.83 - 62.50 = 23.33
Lightweight CNN | finetuned_target_3class: Gap(A,D) = 98.49 - 83.03 = 15.45; Gap(D,R) = 83.03 - 54.66 = 28.37
MobileNetV2 | finetuned_target_3class: Gap(A,D) = 97.77 - 73.88 = 23.89; Gap(D,R) = 73.88 - 51.67 = 22.21
EfficientNetB0 | source_3class: Gap(A,D) = 99.90 - 93.48 = 6.43; Gap(D,R) = 93.48 - 90.00 = 3.48
Lightweight CNN | source_3class: Gap(A,D) = 99.00 - 86.35 = 12.64; Gap(D,R) = 86.35 - 53.59 = 32.76
MobileNetV2 | source_3class: Gap(A,D) = 99.51 - 81.11 = 18.40; Gap(D,R) = 81.11 - 60.00 = 21.11
EfficientNetB0 | zero_shot_target_3class: Gap(A,D) = 99.82 - 79.88 = 19.94; Gap(D,R) = 79.88 - 49.91 = 29.97
Lightweight CNN | zero_shot_target_3class: Gap(A,D) = 78.64 - 58.89 = 19.74; Gap(D,R) = 58.89 - 57.92 = 0.97
MobileNetV2 | zero_shot_target_3class: Gap(A,D) = 96.72 - 53.53 = 43.19; Gap(D,R) = 53.53 - 34.59 = 18.94


In [66]:
FINAL_EXPORT_NAME = "ICDM_final_all_experiments_outputs"

items_to_export = []

candidate_folders = [
    "models",
    "history",
    "histories",
    "reports",
    "prediction_csvs",
    "cross_dataset_transfer_20260530-210441"
]

for folder in candidate_folders:
    if os.path.exists(folder):
        items_to_export.append(folder)

for item in os.listdir("/kaggle/working"):
    if item.startswith("gradcam_") and os.path.isdir(item):
        items_to_export.append(item)

for item in os.listdir("/kaggle/working"):
    if item.endswith((".csv", ".png", ".json", ".pkl", ".keras")):
        items_to_export.append(item)

# Explicitly include all ablation files anywhere under /kaggle/working
for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if "ablation" in file.lower():
            rel_path = os.path.relpath(os.path.join(root, file), "/kaggle/working")
            items_to_export.append(rel_path)

items_to_export = list(dict.fromkeys(items_to_export))

print("Items to export:")
for item in items_to_export:
    print(item)

zip_name = f"{FINAL_EXPORT_NAME}.zip"

!zip -r {zip_name} {' '.join(items_to_export)}

print("Created final export:", zip_name)

Items to export:
cross_dataset_transfer_20260530-210441
ablation_extended_region_metrics_per_seed.csv
ablation_table_A_standard_two_way_three_way.csv
ablation_decision_mechanism_summary_mean_std.csv
ablation_main_paper_extended_region_table.csv
ablation_threshold_sensitivity_7class_main_paper.csv
ablation_threshold_sensitivity_results.csv
ablation_decision_mechanism_results.csv
ablation_three_way_region_accuracy_gap_table.csv
ablation_decision_mechanism_outputs.zip
  adding: cross_dataset_transfer_20260530-210441/ (stored 0%)
  adding: cross_dataset_transfer_20260530-210441/cross_dataset_transfer_per_seed_results.csv (deflated 77%)
  adding: cross_dataset_transfer_20260530-210441/prediction_csvs/ (stored 0%)
  adding: cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_MobileNetV2_finetuned_target_3class_seed_42.csv (deflated 92%)
  adding: cross_dataset_transfer_20260530-210441/prediction_csvs/test_predictions_MobileNetV2_source_3class_seed_777.csv (deflated 81%)
 

In [ ]:
SOURCE_DATASET_DIR = "/kaggle/input/datasets/faysalmiah1721758/potato-dataset"
TARGET_DATASET_DIR = "/kaggle/input/datasets/shahadhossin567r7455/potato-leaf-disease-dataset/Potato Leaf DIsease"